# 3.1 The two-level map equation

This notebook accompanies Section 3.1 of [*Community Detection with the Map Equation and Infomap: Theory and Applications*](https://doi.org/10.1145/3779648).

The map equation scores a partition by its *codelength*: the average number of bits per step to describe a random walk on the network under that partition. Here you compute the codelength by hand for two partitions of one network, then check the values against Infomap. The code favours readability over speed.

In [ ]:
%matplotlib inline

import infomap
import networkx as nx
import numpy as np

from _support import (
    get_map_equation_example_network,
    plot_network,
    plot_modular_network,
)
from scipy.stats import entropy

In [ ]:
# we use the "standard" map equation example network and plot it below
# the network is weighted and undirected
G, pos = get_map_equation_example_network()

plot_network(G, pos=pos)

The codelength weights each node by how often the random walker visits it. On an undirected network, the visit rate of node $u$ is its strength over the total strength, $p_u = \frac{s_u}{\sum_v s_v}$, where the strength $s_u$ is the summed weight of $u$'s links.

In [ ]:
# first, we save all nodes' strength in a dictionary
node_strengths = {u: G.degree(u, weight="weight") for u in G.nodes()}
node_strengths

In [ ]:
# then, we calculate the nodes' visit rates
p = {u: s_u / sum(node_strengths.values()) for u, s_u in node_strengths.items()}
p

With every node in one module, no codewords are reused and the codelength reduces to the entropy of the visit rates, $L\left(\mathsf{M}_1\right) = H(P) = -\sum_u p_u \log_2 p_u$, which sets the baseline a useful partition has to beat.

In [ ]:
# we can put all nodes into a single module
M_1 = [list(G.nodes())]

# then, the codelength is simply the entropy over the nodes' visit rates
codelength = -sum(p_u * np.log2(p_u) for p_u in p.values())
print(f"L(M_1) = {codelength:.4f} bits")

You can see four modules in the network. Assigning the nodes to them gives a candidate two-level partition.

In [ ]:
M = [
    [0, 1, 2, 3, 4, 5],
    [6, 7, 8, 9, 10, 11, 12],
    [13, 14, 15, 16],
    [17, 18, 19, 20, 21, 22, 23, 24],
]

In [ ]:
# let's colour the network by the four modules
nx.set_node_attributes(G, {u: i for i, m in enumerate(M) for u in m}, "module")

plot_modular_network(G, pos=pos, show_labels=True)

A two-level partition adds an index codebook for steps between modules and one codebook per module. The two-level map equation sums their entropies, weighted by how often the walker uses each codebook,

$$L\left(\mathsf{M}\right) = q_\curvearrowleft H(Q) + \sum_{\mathsf{m} \in \mathsf{M}} p_\mathsf{m}^\circlearrowright H(P_\mathsf{m}).$$

The walker enters module $\mathsf{m}$ at rate $q_{\mathsf{m}\curvearrowleft} = \sum_{u \notin \mathsf{m},\, v \in \mathsf{m}} p_u p_{uv}$, with step probability $p_{uv} = \frac{w_{uv}}{\sum_v w_{uv}}$.

In [ ]:
# first, we compute the transition rates because we will use them
transition_rates = dict()

for u in G.nodes:
    for v in G.neighbors(u):
        transition_rates[(u, v)] = G.get_edge_data(u, v)["weight"] / node_strengths[u]

transition_rates

In [ ]:
# to calculate the two-level map equation, we need to know the module entry rates
entry_rates = []

# for each module, sum the flow entering it from nodes outside the module
for m in M:
    m_entry = 0
    for v in m:
        for u in G.neighbors(v):
            if u not in m:
                m_entry += p[u] * transition_rates[u, v]
    entry_rates.append(m_entry)

entry_rates

In [ ]:
# then, we're ready to calculate the codelength
codelength = 0

# first, the codelength for the index-level codebook.
Q = entry_rates
q = sum(Q)
codelength += q * entropy(Q, base=2)

# then, we add the codelength for each module
for m, m_exit in zip(M, entry_rates):
    P_m = [p[u] for u in m] + [m_exit]
    p_m = sum(P_m)
    codelength += p_m * entropy(P_m, base=2)

print(f"L(M) = {codelength:.4f} bits")

Infomap optimizes the same map equation. Running it should recover the four modules and the codelength you computed by hand.

In [ ]:
# we can use Infomap to compute the codelength for the one-level partition by using the no_infomap flag.

# run Infomap with no_infomap to skip optimization and read the one-level codelength
result = infomap.run(G, options=infomap.Options(no_infomap=True))

print(f"L(M_1) = {result.codelength:.4f} bits")

In [ ]:
# or we can use Infomap to detect communities
result = infomap.run(G, seed=123)

# collect Infomap's modules
modules = dict()
for u, m in result.modules().items():
    if m not in modules:
        modules[m] = set()
    modules[m].add(u)

print(f"Infomap detected modules: {[sorted(m) for m in modules.values()]}")
print(f"L(M) = {result.codelength:.4f} bits")

In [ ]:
# Maintenance smoke assertions
G, _ = get_map_equation_example_network()
assert G.number_of_nodes() == 25
assert G.number_of_edges() == 43
assert infomap.Infomap() is not None